In [7]:
import pandas as pd

from model_training.common.history_prep import prepare_history_df
from model_training.common.time_split import time_split
from model_training.config import PATH_GAMLOGS_COMBINED
from model_training.utils.team_codes import norm_team
filesss = "../data/all_gamelogs_combined.csv"

df = pd.read_csv(filesss, low_memory=False)
df = prepare_history_df(df, norm_team_fn=norm_team)

print(df.columns.tolist())
print(df[["game_date", "player", "team", "opp", "season", "mp_minutes"]].head())

train_df, valid_df = time_split(df, split_date="2025-01-01", date_col="game_date")
print(len(train_df), len(valid_df))
print(train_df["game_date"].max(), valid_df["game_date"].min())

['player', 'date', 'season', 'mp', 'team', 'opp', 'fg', 'fga', 'fg3', 'fg3a', 'ft', 'fta', 'orb', 'drb', 'trb', 'ast', 'stl', 'blk', 'tov', 'pf', 'pts', 'is_home', 'is_win', 'mp_minutes', 'usage', 'game_date', 'reb', 'fg3m', 'starter_flag']
   game_date      player team  opp  season  mp_minutes
0 2022-10-20  A.J. Green  MIL  PHI    2023         0.0
1 2022-10-22  A.J. Green  MIL  HOU    2023         0.0
2 2022-10-26  A.J. Green  MIL  BRK    2023         0.0
3 2022-10-28  A.J. Green  MIL  NYK    2023         0.0
4 2022-10-29  A.J. Green  MIL  ATL    2023         0.0
38442 48030
2024-12-31 00:00:00 2025-01-01 00:00:00


In [15]:
import pandas as pd
from ticket.score_leg import score_legs

df = pd.concat([
    pd.read_csv("../results/2026-03-19/pred_ast.csv"),
    pd.read_csv("../results/2026-03-19/pred_reb.csv"),
    pd.read_csv("../results/2026-03-19/pred_fg3.csv"),
    pd.read_csv("../results/2026-03-19/pred_pts.csv"),
], ignore_index=True)

scored = score_legs(df)

print("Columns added:")
print([c for c in scored.columns if "score" in c or "can_" in c or "edge" in c])

print()
print(scored.sort_values("score_safe", ascending=False)[[
    "player", "stat", "line", "side", "p_hit", "score_safe", "can_safe"
]].head(20))

print()
print(scored.sort_values("score_balanced", ascending=False)[[
    "player", "stat", "line", "side", "p_hit", "score_balanced", "can_balanced"
]].head(20))

print()
print(scored.sort_values("score_lotto", ascending=False)[[
    "player", "stat", "line", "side", "p_hit", "score_lotto", "can_lotto"
]].head(20))

Columns added:
['edge_raw', 'edge_norm', 'score']



KeyError: 'score_safe'

In [17]:
import pandas as pd
from pathlib import Path

latest = sorted([p for p in Path("../results").iterdir() if p.is_dir()])[-1]

df = pd.concat([
    pd.read_csv(latest / "pred_pts.csv"),
    pd.read_csv(latest / "pred_reb.csv"),
    pd.read_csv(latest / "pred_ast.csv"),
    pd.read_csv(latest / "pred_fg3.csv"),
])

print("TOTAL ROWS:", len(df))
print("ELIGIBLE:", (df["is_eligible"] == 1).sum())
print(df[df["is_eligible"] == 1].head())

TOTAL ROWS: 430
ELIGIBLE: 270
    game_date               player team  opp stat  pred_mean  baseline_mean  \
0  2026-02-24  Dominick Barlow(TW)  PHI  IND  pts   9.207601       8.369565   
4  2026-02-24      Kelly Oubre Jr.  PHI  IND  pts  15.744809      16.928060   
7  2026-02-24       Quentin Grimes  PHI  IND  pts  11.888546      12.803922   
8  2026-02-24         Tyrese Maxey  PHI  IND  pts  28.402188      31.618177   
9  2026-02-24         VJ Edgecombe  PHI  IND  pts  15.489212      17.282251   

   delta_mean  minutes_proj dist_name  dispersion  is_eligible  \
0    0.838035     21.913333    nbinom    0.060562            1   
4   -1.183251     33.330000    nbinom    0.060562            1   
7   -0.915375     16.456667    nbinom    0.060562            1   
8   -3.215989     35.006666    nbinom    0.060562            1   
9   -1.793039     34.256667    nbinom    0.060562            1   

  eligibility_reason    model_name model_version  
0                 ok  pts_composed            v